# Silver SIDRA — MVP municipal de São Paulo

Lê as sete tabelas Delta da Bronze, padroniza códigos e nomes, mantém população anual e indicadores de saneamento dos Censos de 2010/2022. A Bronze continua nacional. A Silver contém somente SP.

**Grão:** `ano` + `codigo_municipio` (seis dígitos). `municipio` é apenas descrição. Anos 2000, 2007 e 2023 ficam sem população neste MVP. A tabela final não define ainda o esquema estrela.

In [0]:
%run ./00_padronizar_municipios

## Ler as sete fontes da Bronze

In [0]:
from pyspark.sql import functions as F

BRONZE = "dengue_sus.bronze"
SILVER = "dengue_sus.silver"
FONTES = {
    136: "2010_populacao_censo",
    4709: "2022_populacao_censo",
    6579: "populacao_residente_estimada",
    3218: "2010_senamento",  # Nome publicado pelo notebook 01 =(.
    6803: "2022_abastecimento_agua",
    6805: "2022_tipo_de_esgotamento",
    6892: "2022_destino_do_lixo",
}
ANOS_ESTIMATIVAS = [
    2001, 2002, 2003, 2004, 2005, 2006, 2008, 2009,
    2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021,
]
ANOS = set(ANOS_ESTIMATIVAS + [2010, 2022])

fontes = {}
for tabela, nome in FONTES.items():
    origem = f"{BRONZE}.`{nome}`"
    if not spark.catalog.tableExists(origem):
        raise ValueError(f"Tabela Bronze ausente: {origem}. Execute primeiro o notebook 01.")
    bruto = spark.table(origem)
    fontes[tabela] = padronizar_municipios(bruto)
    for coluna in ("ano", "valor"):
        if coluna not in bruto.columns:
            raise ValueError(f"{origem}: coluna {coluna} ausente.")
    invalidos = fontes[tabela].filter(F.col("codigo_municipio").isNull()).count()
    print(f"SIDRA {tabela}: códigos municipais inválidos ou ausentes na Bronze: {invalidos}")


def recortar_sp(df):
    return df.filter(F.col("codigo_municipio").startswith("35"))


def valor_numerico(tipo):
    return F.expr(
        "try_cast(case when trim(valor) = '-' then '0' "
        f"else trim(valor) end as {tipo})"
    )


def exigir_unicidade(df, colunas, origem):
    if df.filter(F.expr(" OR ".join(f"`{c}` IS NULL" for c in colunas))).limit(1).count():
        raise ValueError(f"{origem}: chave nula em {colunas}.")
    if df.groupBy(*colunas).count().filter("count > 1").limit(1).count():
        raise ValueError(f"{origem}: chave duplicada em {colunas}.")

## População municipal

A tabela 136 deve conter somente a categoria Total (`cor_ou_raca_codigo = 0`). As estimativas usam os 19 anos selecionados na Bronze; 2010 e 2022 vêm dos Censos. O símbolo `-` do SIDRA significa zero; outros marcadores não numéricos tornam-se nulos e bloqueiam a publicação de denominadores.

In [0]:
if "cor_ou_raca_codigo" not in fontes[136].columns:
    raise ValueError("SIDRA 136 sem cor_ou_raca_codigo; não é possível validar o Total.")
if fontes[136].filter(
    F.col("cor_ou_raca_codigo").isNull()
    | (F.trim(F.col("cor_ou_raca_codigo")) != "0")
).limit(1).count():
    raise ValueError("SIDRA 136 contém categoria diferente de Total (c86=0).")

populacoes = []
for tabela, anos_permitidos in ((136, {2010}), (4709, {2022}), (6579, set(ANOS_ESTIMATIVAS))):
    quadro = recortar_sp(fontes[tabela]).withColumn("ano", F.col("ano").cast("int"))
    if quadro.filter(~F.col("ano").isin(list(anos_permitidos))).limit(1).count():
        raise ValueError(f"SIDRA {tabela}: anos inesperados na Bronze.")
    populacoes.append(
        quadro.withColumn("populacao", valor_numerico("bigint"))
        .withColumn("fonte_populacao", F.lit(f"SIDRA {tabela}"))
        .select(
            "ano", "codigo_municipio", "codigo_municipio_original",
            F.col("municipio_padronizado").alias("municipio"),
            "municipio_original", "populacao", "fonte_populacao",
        )
    )

populacao = populacoes[0]
for quadro in populacoes[1:]:
    populacao = populacao.unionByName(quadro)

exigir_unicidade(populacao, ["ano", "codigo_municipio"], "população SP")
if populacao.filter(F.col("populacao").isNull() | (F.col("populacao") <= 0)).limit(1).count():
    raise ValueError("População ausente, não numérica ou não positiva em SP.")
contagens_populacao = {linha["ano"]: linha["count"] for linha in populacao.groupBy("ano").count().collect()}
if contagens_populacao != {ano: 645 for ano in ANOS}:
    raise ValueError(f"Grade populacional SP incompleta: {contagens_populacao}")
print("População SP: 645 municípios em cada um dos 21 anos disponíveis.")

## Saneamento por categoria

As categorias abaixo acompanham `utils/analysis_pipeline.py`. Totais e outras categorias da Bronze não entram na Silver. Os prefixos `c2010_` e `c2022_` mantêm as definições dos dois Censos distintas. O valor é percentual.

In [0]:
WATER_CATEGORIES = {
    "92853": "pct_agua_rede_geral",
    "10971": "pct_agua_poco_nascente_propriedade",
    "121290": "pct_agua_poco_nascente_fora_propriedade",
    "121294": "pct_agua_rio_acude_lago_igarape",
    "121296": "pct_agua_poco_nascente_aldeia",
    "121297": "pct_agua_poco_nascente_fora_aldeia",
    "121295": "pct_agua_outra_forma",
}

SANITATION_CATEGORIES = {
    "9678": "pct_banheiro_exclusivo_rede_ou_fossa_septica",
    "2950": "pct_banheiro_exclusivo_outro_escoadouro",
    "9679": "pct_sanitario_rede_ou_fossa_septica",
    "2964": "pct_sanitario_outro_escoadouro",
    "10006": "pct_sem_banheiro_ou_sanitario",
}

WASTE_CATEGORIES = {
    "92863": "pct_lixo_servico_limpeza",
    "92864": "pct_lixo_cacamba_servico_limpeza",
    "1091": "pct_lixo_outro_destino",
}

ENERGY_CATEGORIES = {
    "3011": "pct_com_energia_eletrica",
    "3018": "pct_sem_energia_eletrica",
}

WATER_2022_CATEGORIES = {
    "72144": "pct_agua_rede_geral_principal",
    "72146": "pct_agua_ligada_rede_poco_profundo_principal",
    "72147": "pct_agua_ligada_rede_poco_raso_principal",
    "72148": "pct_agua_ligada_rede_nascente_principal",
    "72149": "pct_agua_ligada_rede_carro_pipa_principal",
    "72150": "pct_agua_ligada_rede_chuva_principal",
    "72151": "pct_agua_ligada_rede_superficial_principal",
    "72152": "pct_agua_ligada_rede_outra_principal",
    "72154": "pct_agua_sem_rede_poco_profundo",
    "72155": "pct_agua_sem_rede_poco_raso",
    "72156": "pct_agua_sem_rede_nascente",
    "72157": "pct_agua_sem_rede_carro_pipa",
    "72158": "pct_agua_sem_rede_chuva",
    "72159": "pct_agua_sem_rede_superficial",
    "72160": "pct_agua_sem_rede_outra",
}

SANITATION_2022_CATEGORIES = {
    "72110": "pct_esgoto_rede_geral_ou_pluvial",
    "72111": "pct_esgoto_fossa_ligada_rede",
    "72112": "pct_esgoto_fossa_nao_ligada_rede",
    "72113": "pct_esgoto_fossa_rudimentar",
    "92858": "pct_esgoto_vala",
    "72114": "pct_esgoto_rio_lago_corrego_mar",
    "72115": "pct_esgoto_outra_forma",
    "92861": "pct_sem_banheiro_ou_sanitario",
}

WASTE_2022_CATEGORIES = {
    "72120": "pct_lixo_coletado_domicilio",
    "72121": "pct_lixo_cacamba",
    "72122": "pct_lixo_queimado_propriedade",
    "72123": "pct_lixo_enterrado_propriedade",
    "72124": "pct_lixo_terreno_encosta_area_publica",
    "1091": "pct_lixo_outro_destino",
}

In [0]:
CATEGORIAS_2010 = {
    "agua": ("forma_de_abastecimento_de_agua_codigo", WATER_CATEGORIES),
    "esgotamento": (
        "existencia_de_banheiro_ou_sanitario_e_esgotamento_sanitario_codigo",
        SANITATION_CATEGORIES,
    ),
    "lixo": ("destino_do_lixo_codigo", WASTE_CATEGORIES),
    "energia": ("existencia_de_energia_eletrica_codigo", ENERGY_CATEGORIES),
}
CATEGORIAS_2022 = {
    6803: ("agua", "existencia_de_ligacao_a_rede_geral_de_distribuicao_de_agua_e_principal_forma_de_abastecimento_de_agua_codigo", WATER_2022_CATEGORIES),
    6805: ("esgotamento", "tipo_de_esgotamento_sanitario_codigo", SANITATION_2022_CATEGORIES),
    6892: ("lixo", "destino_do_lixo_codigo", WASTE_2022_CATEGORIES),
}


def selecionar_indicadores(tabela, ano, dimensao, coluna_categoria, mapa, quadro):
    if coluna_categoria not in quadro.columns:
        raise ValueError(f"SIDRA {tabela}: coluna de categoria ausente: {coluna_categoria}")
    categoria = F.trim(F.col(coluna_categoria))
    pares = [F.lit(item) for par in mapa.items() for item in par]
    rotulos = F.create_map(*pares)
    selecionado = (
        recortar_sp(quadro)
        .filter(categoria.isin(list(mapa)))
        .withColumn("ano", F.col("ano").cast("int"))
        .withColumn("categoria_codigo", categoria)
        .withColumn("indicador", F.concat(F.lit(f"c{ano}_"), F.element_at(rotulos, categoria)))
        .withColumn("valor_numerico", valor_numerico("double"))
        .select(
            "ano", "codigo_municipio", "codigo_municipio_original",
            F.col("municipio_padronizado").alias("municipio"),
            "municipio_original", F.lit(tabela).alias("tabela_sidra"),
            F.lit(dimensao).alias("dimensao"), "categoria_codigo", "indicador",
            F.col("valor").alias("valor_original"), F.col("valor_numerico").alias("valor"),
        )
    )
    if selecionado.filter(F.col("ano") != ano).limit(1).count():
        raise ValueError(f"SIDRA {tabela}: ano inesperado nas categorias de {dimensao}.")
    return selecionado


longos = []
for dimensao, (coluna, mapa) in CATEGORIAS_2010.items():
    if "dimensao" not in fontes[3218].columns:
        raise ValueError("SIDRA 3218 sem a coluna dimensao criada na Bronze.")
    quadro = fontes[3218].filter(F.col("dimensao") == dimensao)
    longos.append(selecionar_indicadores(3218, 2010, dimensao, coluna, mapa, quadro))
for tabela, (dimensao, coluna, mapa) in CATEGORIAS_2022.items():
    longos.append(selecionar_indicadores(tabela, 2022, dimensao, coluna, mapa, fontes[tabela]))

saneamento_longo = longos[0]
for quadro in longos[1:]:
    saneamento_longo = saneamento_longo.unionByName(quadro)

exigir_unicidade(saneamento_longo, ["ano", "codigo_municipio", "indicador"], "saneamento SP")
if saneamento_longo.filter(F.col("valor").isNull() | ~F.col("valor").between(0, 100)).limit(1).count():
    raise ValueError("Saneamento SP: percentual ausente, não numérico ou fora de [0, 100].")

for ano, dimensao, mapa in (
    [(2010, dim, config[1]) for dim, config in CATEGORIAS_2010.items()]
    + [(2022, config[0], config[2]) for config in CATEGORIAS_2022.values()]
):
    quadro = saneamento_longo.filter((F.col("ano") == ano) & (F.col("dimensao") == dimensao))
    total = quadro.count()
    if total != 645 * len(mapa):
        raise ValueError(f"Saneamento {ano}/{dimensao}: {total} linhas; esperadas {645 * len(mapa)}.")
    codigos = quadro.select("codigo_municipio").distinct()
    grade = populacao.filter(F.col("ano") == ano).select("codigo_municipio")
    if codigos.join(grade, "codigo_municipio", "left_anti").limit(1).count() or grade.join(
        codigos, "codigo_municipio", "left_anti"
    ).limit(1).count():
        raise ValueError(f"Saneamento {ano}/{dimensao}: municípios fora da grade populacional.")
    desvios = (
        quadro.groupBy("codigo_municipio")
        .agg(F.sum("valor").alias("total_percentual"))
        .filter(F.abs(F.col("total_percentual") - 100) > 0.1)
    )
    if desvios.limit(1).count():
        raise ValueError(f"Saneamento {ano}/{dimensao}: categorias não fecham 100%.")

print("Saneamento SP: categorias, unicidade e fechamento percentual validados.")

## Agregação por município e ano

A população define a grade anual. Cada indicador de saneamento vira uma coluna própria e permanece nulo fora do respectivo Censo. O join é por código e ano; nomes não participam da ligação.

In [0]:
indicadores = sorted(
    [f"c2010_{nome}" for _, mapa in CATEGORIAS_2010.values() for nome in mapa.values()]
    + [f"c2022_{nome}" for _, _, mapa in CATEGORIAS_2022.values() for nome in mapa.values()]
)
saneamento_municipal = (
    saneamento_longo.groupBy("ano", "codigo_municipio")
    .pivot("indicador", indicadores)
    .agg(F.first("valor"))
)
agregado = populacao.join(saneamento_municipal, ["ano", "codigo_municipio"], "left")
exigir_unicidade(agregado, ["ano", "codigo_municipio"], "SIDRA municipal SP")
if agregado.count() != 645 * len(ANOS):
    raise ValueError("Agregação alterou a grade populacional de SP.")
print(f"Agregado SP: {645 * len(ANOS):,} linhas e {len(indicadores)} indicadores de saneamento.")

## Publicar as três tabelas Silver

As verificações ocorrem antes da escrita. Uma tabela existente com formato, tipo ou colunas incompatíveis interrompe a execução. A reexecução substitui o mesmo recorte, sem acrescentar linhas.

In [0]:
saidas = {
    "sidra_populacao_sp": populacao,
    "sidra_saneamento_sp_longo": saneamento_longo,
    "sidra_municipio_ano_sp": agregado,
}
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
for nome, quadro in saidas.items():
    destino = f"{SILVER}.{nome}"
    if spark.catalog.tableExists(destino):
        existente = spark.catalog.getTable(destino)
        detalhe = spark.sql(f"DESCRIBE DETAIL {destino}").first()
        if existente.tableType != "MANAGED" or detalhe["format"] != "delta":
            raise ValueError(f"{destino}: destino existente não é Delta gerenciado.")
        if spark.table(destino).dtypes != quadro.dtypes:
            raise ValueError(f"{destino}: esquema existente incompatível; confira antes de substituir.")

for nome, quadro in saidas.items():
    destino = f"{SILVER}.{nome}"
    quadro.write.format("delta").mode("overwrite").saveAsTable(destino)
    print(f"Publicado: {destino} ({spark.table(destino).count():,} linhas)")

display(
    spark.table(f"{SILVER}.sidra_municipio_ano_sp")
    .select("ano", "codigo_municipio", "municipio", "populacao")
    .groupBy("ano").count().orderBy("ano")
)
display(
    spark.table(f"{SILVER}.sidra_municipio_ano_sp")
    .filter((F.col("codigo_municipio") == "355030") & F.col("ano").isin(2010, 2022))
    .select("ano", "codigo_municipio", "municipio", "populacao", indicadores[0], indicadores[-1])
    .orderBy("ano")
)